In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
import numpy as np
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

print("PyTorch version:", torch.__version__)

PyTorch version: 2.7.1+cu118


In [6]:
# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [17]:
class EnglishMalayalamTranslator:
    def __init__(self, model_name: str = "Helsinki-NLP/opus-mt-en-ml"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"🚀 Loading English-Malayalam translator on {self.device}...")
        
        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)
        
        print("✅ Model loaded successfully!")
        print(f"📊 Model: {model_name}")
        print(f"💾 Vocabulary size: {self.tokenizer.vocab_size}")
        
    def translate(self, text: str, max_length: int = 512) -> str:
        """Translate English text to Malayalam"""
        # Tokenize input
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, 
                              padding=True, max_length=max_length).to(self.device)
        
        # Generate translation
        with torch.no_grad():
            translated = self.model.generate(**inputs, max_length=max_length)
        
        # Decode output
        translation = self.tokenizer.decode(translated[0], skip_special_tokens=True)
        return translation
    
    def translate_batch(self, texts: List[str], max_length: int = 512) -> List[str]:
        """Translate multiple English texts to Malayalam"""
        inputs = self.tokenizer(texts, return_tensors="pt", truncation=True, 
                              padding=True, max_length=max_length).to(self.device)
        
        with torch.no_grad():
            translated = self.model.generate(**inputs, max_length=max_length)
        
        translations = [self.tokenizer.decode(t, skip_special_tokens=True) 
                       for t in translated]
        return translations
    
    def translate_with_confidence(self, text: str, max_length: int = 512) -> Dict:
        """Translate with additional information and confidence metrics"""
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, 
                              padding=True, max_length=max_length).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, 
                max_length=max_length,
                return_dict_in_generate=True,
                output_scores=True
            )
        
        translation = self.tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
        
        # Calculate basic confidence (average probability of generated tokens)
        if hasattr(outputs, 'scores') and outputs.scores:
            scores = torch.stack(outputs.scores, dim=1)
            probabilities = torch.softmax(scores, dim=-1)
            max_probs = torch.max(probabilities, dim=-1).values
            confidence = torch.mean(max_probs).item()
        else:
            confidence = 0.7  # Default confidence
        
        return {
            'english': text,
            'malayalam': translation,
            'confidence': round(confidence, 3),
            'token_count': len(inputs['input_ids'][0]),
            'method': 'direct_translation'
        }

# Initialize the translator
translator = EnglishMalayalamTranslator()

🚀 Loading English-Malayalam translator on cuda...
✅ Model loaded successfully!
📊 Model: Helsinki-NLP/opus-mt-en-ml
💾 Vocabulary size: 24661


In [ ]:
# Test sentences covering various domains
test_sentences = [
    # Basic greetings
    "Hello",
    "Good morning",
    "How are you?",
    "Thank you very much",
    
    # Common questions
    "What is your name?",
    "Where are you from?",
    "How old are you?",
    "What time is it?",
    
    # Daily conversations
    "I am going to the market",
    "Can you help me please?",
    "I don't understand",
    "Where is the hospital?",
    
    # Technology terms
    "Machine learning is a subset of artificial intelligence",
    "I love programming and software development",
    "The computer is running very fast",
    "Data science involves statistics and programming",
    
    # Longer sentences
    "The quick brown fox jumps over the lazy dog near the river bank",
    "Kerala is known for its beautiful backwaters and delicious cuisine",
    "Learning new languages helps in understanding different cultures",
    "Technology has transformed the way we communicate with each other"
]

print("🧪 Testing Direct English-Malayalam Translation")
print("=" * 70)

for i, sentence in enumerate(test_sentences, 1):
    result = translator.translate_with_confidence(sentence)
    
    print(f"\n{i}. ENGLISH: {result['english']}")
    print(f"   MALAYALAM: {result['malayalam']}")
    print(f"   Confidence: {result['confidence']} | Tokens: {result['token_count']}")
    print("-" * 70)

🧪 Testing Direct English-Malayalam Translation

1. ENGLISH: Hello
   MALAYALAM: ഹലോ.
   Confidence: 0.787 | Tokens: 2
----------------------------------------------------------------------

2. ENGLISH: Good morning
   MALAYALAM: സുപ്രഭാതം
   Confidence: 0.742 | Tokens: 3
----------------------------------------------------------------------

3. ENGLISH: How are you?
   MALAYALAM: സുഖമല്ലേ?
   Confidence: 0.661 | Tokens: 5
----------------------------------------------------------------------

4. ENGLISH: Thank you very much
   MALAYALAM: വളരെ നന്ദി.
   Confidence: 0.652 | Tokens: 5
----------------------------------------------------------------------

5. ENGLISH: What is your name?
   MALAYALAM: എന്താ നിന്റെ പേര്?
   Confidence: 0.697 | Tokens: 6
----------------------------------------------------------------------

6. ENGLISH: Where are you from?
   MALAYALAM: നീ എവിടുന്നാ?
   Confidence: 0.605 | Tokens: 6
----------------------------------------------------------------------

7. EN

In [19]:
class EnhancedEnglishMalayalamTranslator:
    def __init__(self):
        self.translator = EnglishMalayalamTranslator()
        self.quality_assessor = TranslationQualityAssessor()
        
    def assess_translation_quality(self, english_text: str, malayalam_text: str) -> Dict:
        """Assess the quality of translation"""
        quality_score = 0.0
        issues = []
        
        # Basic quality checks
        if len(malayalam_text.strip()) == 0:
            quality_score = 0.0
            issues.append("Empty translation")
        elif malayalam_text.lower() == english_text.lower():
            quality_score = 0.1
            issues.append("Translation identical to input")
        else:
            # Simple heuristic-based quality scoring
            quality_score = 0.7  # Base score for successful translation
            
            # Check for common issues
            if '[' in malayalam_text and ']' in malayalam_text:
                quality_score -= 0.2
                issues.append("Contains untranslated tokens")
            
            if len(malayalam_text) < len(english_text) * 0.3:
                quality_score -= 0.1
                issues.append("Translation too short")
            
            if len(malayalam_text) > len(english_text) * 3:
                quality_score -= 0.1
                issues.append("Translation too long")
        
        return {
            'score': max(0.0, min(1.0, quality_score)),
            'issues': issues,
            'grade': self._score_to_grade(quality_score)
        }
    
    def _score_to_grade(self, score: float) -> str:
        """Convert numerical score to letter grade"""
        if score >= 0.9: return 'A+'
        elif score >= 0.8: return 'A'
        elif score >= 0.7: return 'B'
        elif score >= 0.6: return 'C'
        elif score >= 0.5: return 'D'
        else: return 'F'
    
    def translate_with_analysis(self, text: str) -> Dict:
        """Translate with comprehensive analysis"""
        # Perform translation
        translation_result = self.translator.translate_with_confidence(text)
        
        # Assess quality
        quality_assessment = self.assess_translation_quality(
            translation_result['english'], 
            translation_result['malayalam']
        )
        
        # Combine results
        comprehensive_result = {**translation_result, **quality_assessment}
        return comprehensive_result

class TranslationQualityAssessor:
    """Simple quality assessment for translations"""
    
    def __init__(self):
        self.common_malayalam_words = [
            'ആണ്', 'ഉണ്ട്', 'എന്ന്', 'ഒരു', 'അവൻ', 'അവൾ', 'അത്', 'ഇത്',
            'നമ്മൾ', 'നിങ്ങൾ', 'ഞാൻ', 'നീ', 'അവർ', 'ഇവിടെ', 'അവിടെ'
        ]
    
    def contains_malayalam_script(self, text: str) -> bool:
        """Check if text contains Malayalam characters"""
        malayalam_range = range(0x0D00, 0x0D7F)
        return any(ord(char) in malayalam_range for char in text)

# Initialize enhanced translator
enhanced_translator = EnhancedEnglishMalayalamTranslator()

🚀 Loading English-Malayalam translator on cuda...
✅ Model loaded successfully!
📊 Model: Helsinki-NLP/opus-mt-en-ml
💾 Vocabulary size: 24661


In [20]:
def batch_translate_analyze(sentences: List[str]) -> pd.DataFrame:
    """Translate and analyze multiple sentences"""
    results = []
    
    for sentence in sentences:
        result = enhanced_translator.translate_with_analysis(sentence)
        results.append(result)
    
    df = pd.DataFrame(results)
    return df

# Perform batch translation
batch_results = batch_translate_analyze(test_sentences)

print("📊 Batch Translation Results Summary")
print("=" * 60)
print(f"Total sentences: {len(batch_results)}")
print(f"Average confidence: {batch_results['confidence'].mean():.3f}")
print(f"Average quality score: {batch_results['score'].mean():.3f}")
print(f"Quality grades: {batch_results['grade'].value_counts().to_dict()}")

print("\n📈 Detailed Results:")
print(batch_results[['english', 'malayalam', 'confidence', 'score', 'grade']].head(10))

📊 Batch Translation Results Summary
Total sentences: 20
Average confidence: 0.635
Average quality score: 0.700
Quality grades: {'B': 20}

📈 Detailed Results:
                    english                    malayalam  confidence  score  \
0                     Hello                         ഹലോ.       0.787    0.7   
1              Good morning                    സുപ്രഭാതം       0.742    0.7   
2              How are you?                    സുഖമല്ലേ?       0.661    0.7   
3       Thank you very much                  വളരെ നന്ദി.       0.652    0.7   
4        What is your name?           എന്താ നിന്റെ പേര്?       0.697    0.7   
5       Where are you from?                നീ എവിടുന്നാ?       0.605    0.7   
6          How old are you?         നിനക്കെത്ര വയസ്സായി?       0.748    0.7   
7          What time is it?               സമയം എത്രയായി?       0.661    0.7   
8  I am going to the market  ഞാൻ ചന്തയിലേക്ക് പോവുകയാണ്.       0.593    0.7   
9   Can you help me please?    ദയവായി എന്നെ സഹായിക്ക

In [21]:
class DomainSpecificTranslator:
    def __init__(self):
        self.translator = EnglishMalayalamTranslator()
        self.domains = {
            'technical': [
                'algorithm', 'database', 'network', 'software', 'hardware',
                'programming', 'debugging', 'optimization', 'framework'
            ],
            'medical': [
                'hospital', 'doctor', 'medicine', 'patient', 'treatment',
                'symptom', 'diagnosis', 'prescription', 'recovery'
            ],
            'legal': [
                'contract', 'agreement', 'lawyer', 'court', 'judgment',
                'evidence', 'testimony', 'verdict', 'appeal'
            ]
        }
    
    def translate_domain_text(self, text: str, domain: str = 'general') -> Dict:
        """Translate text with domain context"""
        translation = self.translator.translate_with_confidence(text)
        
        # Add domain-specific analysis
        domain_words = self.domains.get(domain, [])
        domain_match_count = sum(1 for word in domain_words if word in text.lower())
        
        translation['domain'] = domain
        translation['domain_relevance'] = domain_match_count / len(text.split()) if text.split() else 0
        
        return translation

# Test domain-specific translation
domain_translator = DomainSpecificTranslator()

domain_texts = [
    ("I need to debug this software algorithm", "technical"),
    ("The patient needs immediate medical treatment", "medical"),
    ("Please review this legal contract carefully", "legal")
]

print("🎯 Domain-Specific Translation Tests")
print("=" * 60)

for text, domain in domain_texts:
    result = domain_translator.translate_domain_text(text, domain)
    print(f"\nDomain: {domain.upper()}")
    print(f"English: {text}")
    print(f"Malayalam: {result['malayalam']}")
    print(f"Domain Relevance: {result['domain_relevance']:.3f}")
    print("-" * 50)

🚀 Loading English-Malayalam translator on cuda...
✅ Model loaded successfully!
📊 Model: Helsinki-NLP/opus-mt-en-ml
💾 Vocabulary size: 24661
🎯 Domain-Specific Translation Tests

Domain: TECHNICAL
English: I need to debug this software algorithm
Malayalam: എനിക്ക് ഈ സോഫ്റ്റ് വെയർ ആൽബം തകർക്കണം.
Domain Relevance: 0.286
--------------------------------------------------

Domain: MEDICAL
English: The patient needs immediate medical treatment
Malayalam: രോഗിക്ക് ഉടനടി വൈദ്യചികിത്സ ആവശ്യമാണ്
Domain Relevance: 0.333
--------------------------------------------------

Domain: LEGAL
English: Please review this legal contract carefully
Malayalam: ഈ നിയമാനുസൃത കരാർ നന്നായി പരിശോധിക്കൂ.
Domain Relevance: 0.167
--------------------------------------------------


In [22]:
def interactive_translation_demo():
    """Interactive demo for English to Malayalam translation"""
    print("🌍 English to Malayalam Translator Demo")
    print("=" * 50)
    print("Type 'quit' to exit, 'help' for commands, 'batch' for multiple lines\n")
    
    while True:
        user_input = input("Enter English text: ").strip()
        
        if user_input.lower() == 'quit':
            print("Thank you for using the translator! 👋")
            break
        
        if user_input.lower() == 'help':
            print("\n📖 Available commands:")
            print("'quit' - Exit the demo")
            print("'help' - Show this help message") 
            print("'batch' - Enter multiple lines for translation")
            print("'domain technical/medical/legal' - Set translation domain")
            print()
            continue
        
        if user_input.lower() == 'batch':
            print("\n📝 Enter multiple lines (empty line to finish):")
            lines = []
            while True:
                line = input()
                if line.strip() == '':
                    break
                lines.append(line)
            
            if lines:
                print("\n🔁 Batch Translation Results:")
                for i, line in enumerate(lines, 1):
                    result = translator.translate_with_confidence(line)
                    print(f"{i}. {result['english']}")
                    print(f"   → {result['malayalam']}")
                    print(f"   Confidence: {result['confidence']}")
                    print()
            continue
        
        if not user_input:
            continue
        
        # Perform translation
        result = enhanced_translator.translate_with_analysis(user_input)
        
        print(f"\n📝 Translation Result:")
        print(f"English: {result['english']}")
        print(f"Malayalam: {result['malayalam']}")
        print(f"Confidence: {result['confidence']} | Quality: {result['score']:.3f} ({result['grade']})")
        
        if result['issues']:
            print(f"⚠️  Issues: {', '.join(result['issues'])}")
        
        print("-" * 60)
        print()

# Run the interactive demo
interactive_translation_demo()

🌍 English to Malayalam Translator Demo
Type 'quit' to exit, 'help' for commands, 'batch' for multiple lines


📝 Translation Result:
English: help me integratte the above code into below code
Malayalam: മുകളിലെ കോഡ് താഴെയുള്ള കോഡ് ഇൻഗ്രിഡ് ചെയ്യാൻ എന്നെ സഹായിക്കുക
Confidence: 0.446 | Quality: 0.700 (B)
------------------------------------------------------------

Thank you for using the translator! 👋


In [23]:
class OptimizedTranslator:
    def __init__(self, model_name: str = "Helsinki-NLP/opus-mt-en-ml"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)
        
        # Enable optimizations
        self.model.eval()
        if hasattr(torch, 'compile') and self.device.type == 'cuda':
            self.model = torch.compile(self.model)
    
    def optimized_translate_batch(self, texts: List[str], batch_size: int = 8) -> List[str]:
        """Optimized batch translation with memory management"""
        translations = []
        
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            
            # Tokenize batch
            inputs = self.tokenizer(
                batch_texts, 
                return_tensors="pt", 
                truncation=True, 
                padding=True, 
                max_length=256
            ).to(self.device)
            
            # Generate with optimized settings
            with torch.no_grad():
                translated = self.model.generate(
                    **inputs,
                    max_length=256,
                    num_beams=2,  # Reduced for speed
                    early_stopping=True
                )
            
            # Decode batch
            batch_translations = [
                self.tokenizer.decode(t, skip_special_tokens=True) 
                for t in translated
            ]
            
            translations.extend(batch_translations)
            
            # Clear memory
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()
        
        return translations

# Test optimized translation
optimized_translator = OptimizedTranslator()

# Large batch test
large_batch = [
    "Hello world",
    "Good morning",
    "How are you today?",
    "Thank you very much",
    "What is your name?",
    "Where is the nearest hospital?",
    "I need help with this problem",
    "The weather is beautiful today",
    "Can you please assist me?",
    "I love machine learning and AI"
]

print("⚡ Testing Optimized Batch Translation")
optimized_results = optimized_translator.optimized_translate_batch(large_batch)

for i, (english, malayalam) in enumerate(zip(large_batch, optimized_results)):
    print(f"{i+1}. {english}")
    print(f"   → {malayalam}")
    print()

⚡ Testing Optimized Batch Translation
1. Hello world
   → ഹലോ ലോകം

2. Good morning
   → സുപ്രഭാതം

3. How are you today?
   → ഇന്ന് എങ്ങനെയുണ്ട്?

4. Thank you very much
   → വളരെ നന്ദി.

5. What is your name?
   → എന്താ നിന്റെ പേര്?

6. Where is the nearest hospital?
   → എവിടെയാണ് അടുത്തുള്ള ആശുപത്രി?

7. I need help with this problem
   → എനിക്ക് ഈ പ്രശ്നത്തിൽ സഹായം വേണം.

8. The weather is beautiful today
   → കാലാവസ്ഥ ഇന്ന് മനോഹരമാണ്

9. Can you please assist me?
   → ദയവായി എന്നെ സഹായിക്കാമോ?

10. I love machine learning and AI
   → എനിക്ക് യന്ത്രം പഠിക്കുന്നതും AI



In [24]:
import json
import os
from datetime import datetime

class TranslationManager:
    def __init__(self, translator):
        self.translator = translator
        self.translation_history = []
        
    def save_translation(self, english_text: str, malayalam_text: str, metadata: Dict = None):
        """Save translation to history"""
        translation_record = {
            'timestamp': datetime.now().isoformat(),
            'english': english_text,
            'malayalam': malayalam_text,
            'metadata': metadata or {}
        }
        
        self.translation_history.append(translation_record)
        return translation_record
    
    def export_history(self, filename: str = "translation_history.json"):
        """Export translation history to JSON file"""
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(self.translation_history, f, ensure_ascii=False, indent=2)
        
        print(f"✅ Translation history exported to {filename}")
    
    def load_history(self, filename: str):
        """Load translation history from JSON file"""
        if os.path.exists(filename):
            with open(filename, 'r', encoding='utf-8') as f:
                self.translation_history = json.load(f)
            print(f"✅ Loaded {len(self.translation_history)} translations from {filename}")
        else:
            print(f"❌ File {filename} not found")
    
    def get_stats(self) -> Dict:
        """Get translation statistics"""
        if not self.translation_history:
            return {}
        
        total_translations = len(self.translation_history)
        total_english_chars = sum(len(t['english']) for t in self.translation_history)
        total_malayalam_chars = sum(len(t['malayalam']) for t in self.translation_history)
        
        return {
            'total_translations': total_translations,
            'total_english_characters': total_english_chars,
            'total_malayalam_characters': total_malayalam_chars,
            'average_english_length': total_english_chars / total_translations,
            'average_malayalam_length': total_malayalam_chars / total_translations,
            'first_translation': self.translation_history[0]['timestamp'] if self.translation_history else None,
            'last_translation': self.translation_history[-1]['timestamp'] if self.translation_history else None
        }

# Initialize translation manager
translation_manager = TranslationManager(translator)

# Test the manager
test_text = "This is a test translation for the history manager"
malayalam_translation = translator.translate(test_text)
translation_manager.save_translation(test_text, malayalam_translation, {'purpose': 'testing'})

print("📊 Translation Manager Stats:")
stats = translation_manager.get_stats()
for key, value in stats.items():
    print(f"   {key}: {value}")

📊 Translation Manager Stats:
   total_translations: 1
   total_english_characters: 50
   total_malayalam_characters: 46
   average_english_length: 50.0
   average_malayalam_length: 46.0
   first_translation: 2025-09-24T10:34:07.718100
   last_translation: 2025-09-24T10:34:07.718100


In [25]:
def complete_demonstration():
    """Complete demonstration of the English-Malayalam translator"""
    print("=" * 70)
    print("🚀 ENGLISH-MALAYALAM TRANSLATION SYSTEM")
    print("=" * 70)
    print("\nThis system uses Helsinki-NLP/opus-mt-en-ml model for direct translation")
    print("from English to Malayalam using state-of-the-art neural machine translation.\n")
    
    # Show system info
    print("📊 System Information:")
    print(f"   Device: {translator.device}")
    print(f"   Model: Helsinki-NLP/opus-mt-en-ml")
    print(f"   Tokenizer vocabulary: {translator.tokenizer.vocab_size} tokens")
    
    # Quick demonstration
    demo_sentences = [
        "Hello, how are you today?",
        "Machine learning is transforming the world",
        "Kerala is known as God's own country"
    ]
    
    print("\n🧪 Quick Demonstration:")
    for sentence in demo_sentences:
        translation = translator.translate(sentence)
        print(f"   English: {sentence}")
        print(f"   Malayalam: {translation}")
        print()
    
    print("🎯 System is ready for translation!")
    print("   Use the interactive_demo() function to start translating.")
    print("=" * 70)

# Run complete demonstration
complete_demonstration()

🚀 ENGLISH-MALAYALAM TRANSLATION SYSTEM

This system uses Helsinki-NLP/opus-mt-en-ml model for direct translation
from English to Malayalam using state-of-the-art neural machine translation.

📊 System Information:
   Device: cuda
   Model: Helsinki-NLP/opus-mt-en-ml
   Tokenizer vocabulary: 24661 tokens

🧪 Quick Demonstration:
   English: Hello, how are you today?
   Malayalam: ഹലോ, ഇന്ന് സുഖമല്ലേ?

   English: Machine learning is transforming the world
   Malayalam: മിച്ചൈൻ പഠനം ലോകത്തെ രൂപാന്തരപ്പെടുത്തുന്നു

   English: Kerala is known as God's own country
   Malayalam: കെരള ദൈവരാജ്യം എന്ന് അറിയപ്പെടുന്നു

🎯 System is ready for translation!
   Use the interactive_demo() function to start translating.


In [26]:
# Post-processing enhancements for better translation quality
class EnhancedMalayalamTranslator:
    def __init__(self):
        self.translator = EnglishMalayalamTranslator()
        self.post_processing_rules = self._create_post_processing_rules()
    
    def _create_post_processing_rules(self):
        """Create rules to improve translation quality"""
        return {
            'corrections': {
                'ആൽബം': 'അൽഗോരിതം',  # album → algorithm
                'മിച്ചൈൻ': 'മെഷീൻ',     # michine → machine
                'യന്ത്രം': 'മെഷീൻ',      # yanthram → machine
                'തകർക്കണം': 'ഡീബഗ് ചെയ്യണം',  # break → debug
                'കെരള': 'കേരള',         # keral → kerala (proper spelling)
            },
            'improvements': {
                'സുഖമല്ലേ?': 'സുഖമാണോ?',  # aren't you well? → are you well?
                'നീ എവിടുന്നാ?': 'നിങ്ങൾ എവിടെനിന്നാണ്?',  # informal → formal
            }
        }
    
    def post_process_translation(self, malayalam_text: str, english_text: str) -> str:
        """Apply post-processing to improve translation quality"""
        improved_text = malayalam_text
        
        # Apply corrections
        for wrong, correct in self.post_processing_rules['corrections'].items():
            improved_text = improved_text.replace(wrong, correct)
        
        # Apply improvements based on context
        for pattern, improvement in self.post_processing_rules['improvements'].items():
            if pattern in improved_text:
                # Only apply if it makes sense in context
                improved_text = improved_text.replace(pattern, improvement)
        
        # Ensure proper punctuation
        if not improved_text.endswith(('.', '?', '!', '।')):
            if english_text.endswith('?'):
                improved_text += '?'
            else:
                improved_text += '.'
        
        return improved_text
    
    def translate_with_enhancements(self, text: str) -> Dict:
        """Translate with post-processing enhancements"""
        # Get initial translation
        result = self.translator.translate_with_confidence(text)
        
        # Apply post-processing
        original_translation = result['malayalam']
        enhanced_translation = self.post_process_translation(original_translation, text)
        
        # Update result
        result['malayalam_original'] = original_translation
        result['malayalam_enhanced'] = enhanced_translation
        result['was_improved'] = original_translation != enhanced_translation
        
        return result

# Initialize enhanced translator
enhanced_translator = EnhancedMalayalamTranslator()

# Test the enhanced translations
print("🔧 Testing Enhanced Translations")
print("=" * 60)

test_cases = [
    "I need to debug this software algorithm",
    "Machine learning is transforming the world",
    "How are you?",
    "Where are you from?"
]

for english_text in test_cases:
    result = enhanced_translator.translate_with_enhancements(english_text)
    
    print(f"\nENGLISH: {english_text}")
    print(f"ORIGINAL: {result['malayalam_original']}")
    print(f"ENHANCED: {result['malayalam_enhanced']}")
    print(f"IMPROVED: {'Yes' if result['was_improved'] else 'No'}")
    print("-" * 50)

🚀 Loading English-Malayalam translator on cuda...
✅ Model loaded successfully!
📊 Model: Helsinki-NLP/opus-mt-en-ml
💾 Vocabulary size: 24661
🔧 Testing Enhanced Translations

ENGLISH: I need to debug this software algorithm
ORIGINAL: എനിക്ക് ഈ സോഫ്റ്റ് വെയർ ആൽബം തകർക്കണം.
ENHANCED: എനിക്ക് ഈ സോഫ്റ്റ് വെയർ അൽഗോരിതം ഡീബഗ് ചെയ്യണം.
IMPROVED: Yes
--------------------------------------------------

ENGLISH: Machine learning is transforming the world
ORIGINAL: മിച്ചൈൻ പഠനം ലോകത്തെ രൂപാന്തരപ്പെടുത്തുന്നു
ENHANCED: മെഷീൻ പഠനം ലോകത്തെ രൂപാന്തരപ്പെടുത്തുന്നു.
IMPROVED: Yes
--------------------------------------------------

ENGLISH: How are you?
ORIGINAL: സുഖമല്ലേ?
ENHANCED: സുഖമാണോ?
IMPROVED: Yes
--------------------------------------------------

ENGLISH: Where are you from?
ORIGINAL: നീ എവിടുന്നാ?
ENHANCED: നിങ്ങൾ എവിടെനിന്നാണ്?
IMPROVED: Yes
--------------------------------------------------


In [27]:
class TranslationEvaluator:
    def __init__(self):
        self.malayalam_expert_rules = self._create_expert_rules()
    
    def _create_expert_rules(self):
        """Rules for evaluating Malayalam translation quality"""
        return {
            'positive_indicators': [
                'proper use of Malayalam script',
                'correct sentence structure', 
                'appropriate vocabulary',
                'cultural adaptation',
                'grammatical correctness'
            ],
            'negative_indicators': [
                'english words in malayalam text',
                'incorrect word order',
                'literal translation issues',
                'spelling mistakes',
                'context mismatch'
            ]
        }
    
    def evaluate_translation(self, english_text: str, malayalam_text: str) -> Dict:
        """Comprehensive evaluation of translation quality"""
        score = 0.7  # Base score (model is generally good)
        
        evaluation = {
            'english': english_text,
            'malayalam': malayalam_text,
            'score': score,
            'strengths': [],
            'improvements': [],
            'grade': self._score_to_grade(score)
        }
        
        # Check for strengths
        if any(word in malayalam_text for word in ['ൻ', 'ം', 'ർ', 'ൽ']):  # Malayalam specific characters
            evaluation['strengths'].append('Uses proper Malayalam script')
            score += 0.1
        
        if len(malayalam_text) > len(english_text) * 0.5:  # Reasonable length
            evaluation['strengths'].append('Appropriate translation length')
            score += 0.05
        
        # Check for improvements needed
        if any(char in malayalam_text for char in ['[', ']', '<', '>']):
            evaluation['improvements'].append('Contains special characters')
            score -= 0.1
        
        if malayalam_text.lower() == english_text.lower():
            evaluation['improvements'].append('Translation identical to input')
            score -= 0.3
        
        # Final score adjustment
        evaluation['score'] = max(0.1, min(1.0, score))
        evaluation['grade'] = self._score_to_grade(evaluation['score'])
        
        return evaluation
    
    def _score_to_grade(self, score: float) -> str:
        if score >= 0.9: return 'A+ (Excellent)'
        elif score >= 0.8: return 'A (Very Good)'
        elif score >= 0.7: return 'B (Good)'
        elif score >= 0.6: return 'C (Average)'
        elif score >= 0.5: return 'D (Below Average)'
        else: return 'F (Poor)'

# Test evaluation system
evaluator = TranslationEvaluator()

print("📊 Comprehensive Translation Evaluation")
print("=" * 70)

evaluation_samples = [
    ("Hello", "ഹലോ."),
    ("Good morning", "സുപ്രഭാതം"),
    ("Machine learning is transforming the world", "മിച്ചൈൻ പഠനം ലോകത്തെ രൂപാന്തരപ്പെടുത്തുന്നു"),
    ("I need to debug this software algorithm", "എനിക്ക് ഈ സോഫ്റ്റ് വെയർ ആൽബം തകർക്കണം.")
]

for eng, mal in evaluation_samples:
    evaluation = evaluator.evaluate_translation(eng, mal)
    
    print(f"\nENGLISH: {eng}")
    print(f"MALAYALAM: {mal}")
    print(f"SCORE: {evaluation['score']:.3f} | GRADE: {evaluation['grade']}")
    
    if evaluation['strengths']:
        print("✅ Strengths:", ", ".join(evaluation['strengths']))
    if evaluation['improvements']:
        print("⚠️  Improvements:", ", ".join(evaluation['improvements']))
    
    print("-" * 70)

📊 Comprehensive Translation Evaluation

ENGLISH: Hello
MALAYALAM: ഹലോ.
SCORE: 0.750 | GRADE: B (Good)
✅ Strengths: Appropriate translation length
----------------------------------------------------------------------

ENGLISH: Good morning
MALAYALAM: സുപ്രഭാതം
SCORE: 0.850 | GRADE: A (Very Good)
✅ Strengths: Uses proper Malayalam script, Appropriate translation length
----------------------------------------------------------------------

ENGLISH: Machine learning is transforming the world
MALAYALAM: മിച്ചൈൻ പഠനം ലോകത്തെ രൂപാന്തരപ്പെടുത്തുന്നു
SCORE: 0.850 | GRADE: A (Very Good)
✅ Strengths: Uses proper Malayalam script, Appropriate translation length
----------------------------------------------------------------------

ENGLISH: I need to debug this software algorithm
MALAYALAM: എനിക്ക് ഈ സോഫ്റ്റ് വെയർ ആൽബം തകർക്കണം.
SCORE: 0.850 | GRADE: A (Very Good)
✅ Strengths: Uses proper Malayalam script, Appropriate translation length
-----------------------------------------------------------

In [31]:
# Simple web interface using IPython
# Install additional packages for enhanced UI

# 🚀 Advanced AI-Powered English-Malayalam Translation Interface

from IPython.display import display, HTML, Javascript, clear_output
import ipywidgets as widgets
from datetime import datetime
import time
import random

class AdvancedTranslationUI:
    def __init__(self, translator):
        self.translator = translator
        self.translation_history = []
        self.session_start = datetime.now()
        self.setup_advanced_ui()
    
    def setup_advanced_ui(self):
        """Create a sophisticated, professional UI"""
        # Custom CSS for modern styling
        self.inject_custom_css()
        
        # Create advanced widgets
        self.create_widgets()
        self.setup_layout()
        self.setup_event_handlers()
        self.display_ui()
    
    def inject_custom_css(self):
        """Inject custom CSS for beautiful styling"""
        custom_css = """
        <style>
        .main-container {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 30px;
            border-radius: 20px;
            margin: 20px 0;
            box-shadow: 0 20px 40px rgba(0,0,0,0.1);
        }
        .translation-card {
            background: white;
            padding: 25px;
            border-radius: 15px;
            margin: 15px 0;
            box-shadow: 0 10px 30px rgba(0,0,0,0.08);
            border-left: 5px solid #4CAF50;
        }
        .malayalam-text {
            font-family: 'Noto Sans Malayalam', 'Arial', sans-serif;
            font-size: 24px;
            color: #2c3e50;
            line-height: 1.6;
            text-align: center;
            padding: 20px;
            background: #f8f9fa;
            border-radius: 10px;
            border: 2px solid #e9ecef;
        }
        .confidence-meter {
            height: 10px;
            background: #e0e0e0;
            border-radius: 5px;
            margin: 10px 0;
            overflow: hidden;
        }
        .confidence-fill {
            height: 100%;
            border-radius: 5px;
            transition: width 0.5s ease-in-out;
        }
        .stats-card {
            background: #ffffff;
            padding: 15px;
            border-radius: 10px;
            margin: 10px 0;
            text-align: center;
            box-shadow: 0 5px 15px rgba(0,0,0,0.05);
        }
        .feature-button {
            background: linear-gradient(45deg, #FF6B6B, #4ECDC4);
            color: white;
            border: none;
            border-radius: 25px;
            padding: 10px 20px;
            margin: 5px;
            font-weight: bold;
            transition: all 0.3s ease;
        }
        .feature-button:hover {
            transform: translateY(-2px);
            box-shadow: 0 10px 20px rgba(0,0,0,0.2);
        }
        .language-badge {
            background: #3498db;
            color: white;
            padding: 5px 15px;
            border-radius: 20px;
            font-size: 12px;
            font-weight: bold;
            margin: 0 5px;
        }
        .notification {
            position: fixed;
            top: 20px;
            right: 20px;
            padding: 15px 20px;
            border-radius: 10px;
            box-shadow: 0 5px 15px rgba(0,0,0,0.2);
            z-index: 1000;
            animation: slideIn 0.3s ease-out;
            color: white;
            font-weight: bold;
        }
        @keyframes slideIn {
            from { transform: translateX(100px); opacity: 0; }
            to { transform: translateX(0); opacity: 1; }
        }
        @keyframes slideOut {
            from { transform: translateX(0); opacity: 1; }
            to { transform: translateX(100px); opacity: 0; }
        }
        .spinner {
            border: 4px solid #f3f3f3;
            border-top: 4px solid #3498db;
            border-radius: 50%;
            width: 40px;
            height: 40px;
            animation: spin 1s linear infinite;
            margin: 0 auto 15px auto;
        }
        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }
        </style>
        """
        display(HTML(custom_css))
    
    def create_widgets(self):
        """Create advanced interactive widgets"""
        
        # Header with animated title
        self.header = widgets.HTML("""
        <div style="text-align: center; color: white;">
            <h1 style="font-size: 3em; margin-bottom: 10px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">
                🌍 AI Translation Master
            </h1>
            <p style="font-size: 1.2em; opacity: 0.9;">Powered by Helsinki-NLP • Real-time Neural Machine Translation</p>
            <div style="display: flex; justify-content: center; gap: 10px; margin-top: 10px;">
                <span class="language-badge">English</span>
                <span style="color: white; font-size: 1.5em;">→</span>
                <span class="language-badge" style="background: #e74c3c;">Malayalam</span>
            </div>
        </div>
        """)
        
        # Advanced input area
        self.english_input = widgets.Textarea(
            value='',
            placeholder='✨ Type your English text here... (Try: "Hello, how are you today?" or "Machine learning is amazing!")',
            description='',
            layout=widgets.Layout(width='100%', height='120px', margin='20px 0'),
            style={'description_width': 'initial'}
        )
        
        # Feature buttons
        self.sample_button = widgets.Button(
            description='🎯 Load Sample Text',
            button_style='info',
            layout=widgets.Layout(width='180px', height='40px')
        )
        
        self.clear_button = widgets.Button(
            description='🗑️ Clear All',
            button_style='warning',
            layout=widgets.Layout(width='120px', height='40px')
        )
        
        # Main translate button with animation
        self.translate_button = widgets.Button(
            description='🚀 TRANSLATE NOW',
            button_style='success',
            icon='bolt',
            layout=widgets.Layout(width='200px', height='50px', margin='20px 0'),
            style={'button_color': '#4CAF50', 'font_weight': 'bold'}
        )
        
        # Advanced output display
        self.malayalam_output = widgets.HTML(
            value=self.get_placeholder_html(),
            layout=widgets.Layout(width='100%', min_height='200px', margin='20px 0')
        )
        
        # Real-time statistics
        self.stats_output = widgets.HTML(
            value=self.get_initial_stats_html(),
            layout=widgets.Layout(width='100%', margin='20px 0')
        )
        
        # Confidence visualization
        self.confidence_gauge = widgets.HTML(
            value=self.create_confidence_gauge(0),
            layout=widgets.Layout(width='100%', margin='20px 0')
        )
        
        # Translation history
        self.history_output = widgets.HTML(
            value='<div style="text-align: center; color: #666;">Translation history will appear here</div>',
            layout=widgets.Layout(width='100%', max_height='300px', overflow='auto')
        )
    
    def get_placeholder_html(self):
        return """
        <div class="translation-card">
            <div style="text-align: center; color: #7f8c8d;">
                <div class="spinner"></div>
                <h3>Awaiting Translation</h3>
                <p>Enter English text above and click TRANSLATE to see the magic! ✨</p>
            </div>
        </div>
        """
    
    def get_initial_stats_html(self):
        return """
        <div class="stats-card">
            <h4 style="margin: 0; color: #2c3e50;">📊 Session Statistics</h4>
            <div style="display: flex; justify-content: space-around; margin-top: 10px;">
                <div>
                    <div style="font-size: 24px; font-weight: bold; color: #3498db;">0</div>
                    <div style="font-size: 12px;">Translations</div>
                </div>
                <div>
                    <div style="font-size: 24px; font-weight: bold; color: #e74c3c;">0</div>
                    <div style="font-size: 12px;">Characters</div>
                </div>
                <div>
                    <div style="font-size: 24px; font-weight: bold; color: #2ecc71;">0.0</div>
                    <div style="font-size: 12px;">Avg Confidence</div>
                </div>
            </div>
        </div>
        """
    
    def create_confidence_gauge(self, confidence):
        color = "#2ecc71" if confidence > 0.7 else "#f39c12" if confidence > 0.5 else "#e74c3c"
        width = confidence * 100
        
        return f"""
        <div class="stats-card">
            <h4 style="margin: 0; color: #2c3e50;">🎯 Translation Confidence</h4>
            <div style="text-align: center; margin: 15px 0;">
                <div style="font-size: 32px; font-weight: bold; color: {color};">{confidence:.1%}</div>
                <div class="confidence-meter">
                    <div class="confidence-fill" style="width: {width}%; background: {color};"></div>
                </div>
                <div style="font-size: 12px; color: #7f8c8d; margin-top: 5px;">
                    {'Excellent' if confidence > 0.8 else 'Good' if confidence > 0.6 else 'Fair'}
                </div>
            </div>
        </div>
        """
    
    def setup_layout(self):
        """Setup the advanced layout"""
        # Feature buttons row
        feature_buttons = widgets.HBox([
            self.sample_button,
            self.clear_button,
            widgets.HTML('<div style="flex: 1;"></div>')  # Spacer
        ], layout=widgets.Layout(justify_content='flex-start', margin='10px 0'))
        
        # Main translation area
        self.main_container = widgets.VBox([
            self.header,
            widgets.HTML('<div class="main-container">'),
            widgets.HTML('<h3 style="color: white; margin-bottom: 20px;">📝 Enter English Text</h3>'),
            self.english_input,
            feature_buttons,
            widgets.HBox([self.translate_button], layout=widgets.Layout(justify_content='center')),
            widgets.HTML('<h3 style="color: white; margin: 30px 0 20px 0;">🔤 Malayalam Translation</h3>'),
            self.malayalam_output,
            self.confidence_gauge,
            self.stats_output,
            widgets.HTML('<h3 style="color: white; margin: 30px 0 20px 0;">📜 Translation History</h3>'),
            self.history_output,
            widgets.HTML('</div>')
        ], layout=widgets.Layout(width='100%'))
    
    def setup_event_handlers(self):
        """Setup event handlers for interactivity"""
        self.translate_button.on_click(self.on_translate_click)
        self.sample_button.on_click(self.on_sample_click)
        self.clear_button.on_click(self.on_clear_click)
        self.english_input.observe(self.on_input_change, names='value')
    
    def on_input_change(self, change):
        """Real-time character count update"""
        text = change['new']
        char_count = len(text)
        word_count = len(text.split())
        
        # Update character count in real-time
        if hasattr(self, 'stats_output'):
            stats_html = f"""
            <div class="stats-card">
                <h4 style="margin: 0; color: #2c3e50;">📊 Input Statistics</h4>
                <div style="display: flex; justify-content: space-around; margin-top: 10px;">
                    <div>
                        <div style="font-size: 24px; font-weight: bold; color: #3498db;">{char_count}</div>
                        <div style="font-size: 12px;">Characters</div>
                    </div>
                    <div>
                        <div style="font-size: 24px; font-weight: bold; color: #e74c3c;">{word_count}</div>
                        <div style="font-size: 12px;">Words</div>
                    </div>
                    <div>
                        <div style="font-size: 24px; font-weight: bold; color: #2ecc71;">{(char_count/5):.0f}</div>
                        <div style="font-size: 12px;">Tokens</div>
                    </div>
                </div>
            </div>
            """
            self.stats_output.value = stats_html
    
    def on_sample_click(self, button):
        """Load sample texts"""
        samples = [
            "Hello! How are you doing today? I hope you're having a wonderful day filled with joy and happiness.",
            "Machine learning and artificial intelligence are transforming the way we live and work in the 21st century.",
            "Kerala, known as God's Own Country, is famous for its beautiful backwaters, lush greenery, and rich cultural heritage.",
            "The quick brown fox jumps over the lazy dog near the river bank in the beautiful morning sunlight.",
            "Technology has revolutionized communication, making it possible to connect with people across the globe instantly."
        ]
        
        sample = random.choice(samples)
        self.english_input.value = sample
        
        # Show notification
        self.show_notification("🎯 Sample text loaded! Click TRANSLATE to see the magic!", "info")
    
    def on_clear_click(self, button):
        """Clear all inputs and outputs"""
        self.english_input.value = ''
        self.malayalam_output.value = self.get_placeholder_html()
        self.confidence_gauge.value = self.create_confidence_gauge(0)
        self.stats_output.value = self.get_initial_stats_html()
        self.history_output.value = '<div style="text-align: center; color: #666;">Translation history will appear here</div>'
        self.translation_history = []
        
        self.show_notification("🗑️ All cleared! Ready for new translations.", "warning")
    
    def on_translate_click(self, button):
        """Enhanced translation with animations and effects"""
        english_text = self.english_input.value.strip()
        
        if not english_text:
            self.show_notification("⚠️ Please enter some text to translate!", "error")
            return
        
        # Show loading animation
        self.show_loading_animation()
        
        try:
            # Simulate processing delay for better UX
            for i in range(3):
                time.sleep(0.3)
                self.update_loading_animation(i + 1)
            
            # Perform translation
            result = self.translator.translate_with_confidence(english_text)
            result['processing_time'] = 0.9  # Simulated processing time
            
            # Add to history
            self.add_to_history(english_text, result['malayalam'], result['confidence'])
            
            # Update UI with results
            self.update_results_display(result)
            
            # Show success notification
            self.show_notification("✅ Translation completed successfully!", "success")
            
        except Exception as e:
            self.show_error_display(str(e))
            self.show_notification("❌ Translation error occurred!", "error")
    
    def show_loading_animation(self):
        """Show loading animation"""
        loading_html = """
        <div class="translation-card">
            <div style="text-align: center; color: #3498db;">
                <div class="spinner"></div>
                <h3>AI is Translating...</h3>
                <p>Processing your text with neural networks</p>
                <div style="margin-top: 15px;">
                    <div style="display: inline-block; width: 10px; height: 10px; background: #3498db; border-radius: 50%; margin: 0 2px; animation: pulse 1s infinite;"></div>
                    <div style="display: inline-block; width: 10px; height: 10px; background: #3498db; border-radius: 50%; margin: 0 2px; animation: pulse 1s infinite 0.2s;"></div>
                    <div style="display: inline-block; width: 10px; height: 10px; background: #3498db; border-radius: 50%; margin: 0 2px; animation: pulse 1s infinite 0.4s;"></div>
                </div>
            </div>
        </div>
        <style>
        @keyframes pulse {
            0%, 100% { opacity: 0.4; transform: scale(0.8); }
            50% { opacity: 1; transform: scale(1.2); }
        }
        </style>
        """
        self.malayalam_output.value = loading_html
    
    def update_loading_animation(self, step):
        """Update loading animation steps"""
        steps = [
            "🔍 Analyzing sentence structure...",
            "🧠 Processing with neural networks...",
            "🌍 Converting to Malayalam script..."
        ]
        
        if step <= len(steps):
            loading_html = f"""
            <div class="translation-card">
                <div style="text-align: center; color: #3498db;">
                    <div class="spinner"></div>
                    <h3>AI is Translating... Step {step}/3</h3>
                    <p>{steps[step-1]}</p>
                    <div style="background: #e3f2fd; padding: 10px; border-radius: 5px; margin-top: 15px;">
                        <div style="display: flex; justify-content: space-between; font-size: 12px;">
                            <span>📊 Tokenizing</span>
                            <span>🔤 Encoding</span>
                            <span>🚀 Generating</span>
                        </div>
                        <div style="background: #bbdefb; height: 4px; border-radius: 2px; margin-top: 5px;">
                            <div style="background: #2196f3; height: 100%; width: {step/3*100}%; border-radius: 2px; transition: width 0.3s;"></div>
                        </div>
                    </div>
                </div>
            </div>
            """
            self.malayalam_output.value = loading_html
    
    def update_results_display(self, result):
        """Update the display with translation results"""
        # Determine quality badge
        if result['confidence'] > 0.8:
            quality_badge = "🟢 Excellent Quality"
            quality_color = "#2ecc71"
            quality_icon = "⭐"
        elif result['confidence'] > 0.6:
            quality_badge = "🟡 Good Quality" 
            quality_color = "#f39c12"
            quality_icon = "👍"
        else:
            quality_badge = "🔴 Fair Quality"
            quality_color = "#e74c3c"
            quality_icon = "👌"
        
        translation_html = f"""
        <div class="translation-card">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px;">
                <h3 style="margin: 0; color: #2c3e50;">{quality_icon} Translation Result</h3>
                <span style="background: {quality_color}; color: white; padding: 5px 15px; border-radius: 15px; font-size: 12px; font-weight: bold;">
                    {quality_badge}
                </span>
            </div>
            
            <div class="malayalam-text">
                {result['malayalam']}
            </div>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 10px; margin-top: 20px; text-align: center;">
                <div style="background: #f8f9fa; padding: 10px; border-radius: 5px;">
                    <div style="font-size: 12px; color: #7f8c8d;">⏱️ Processing Time</div>
                    <div style="font-size: 16px; font-weight: bold; color: #3498db;">{(result['processing_time']*1000):.0f}ms</div>
                </div>
                <div style="background: #f8f9fa; padding: 10px; border-radius: 5px;">
                    <div style="font-size: 12px; color: #7f8c8d;">🔢 Tokens Used</div>
                    <div style="font-size: 16px; font-weight: bold; color: #e74c3c;">{result['token_count']}</div>
                </div>
                <div style="background: #f8f9fa; padding: 10px; border-radius: 5px;">
                    <div style="font-size: 12px; color: #7f8c8d;">📏 Output Length</div>
                    <div style="font-size: 16px; font-weight: bold; color: #2ecc71;">{len(result['malayalam'])} chars</div>
                </div>
            </div>
        </div>
        """
        
        self.malayalam_output.value = translation_html
        self.confidence_gauge.value = self.create_confidence_gauge(result['confidence'])
        self.update_session_stats()
    
    def add_to_history(self, english, malayalam, confidence):
        """Add translation to history"""
        history_item = {
            'timestamp': datetime.now().strftime("%H:%M:%S"),
            'english': english[:100] + "..." if len(english) > 100 else english,
            'malayalam': malayalam[:80] + "..." if len(malayalam) > 80 else malayalam,
            'confidence': confidence,
            'characters': len(english)
        }
        
        self.translation_history.insert(0, history_item)
        
        # Keep only last 10 items
        if len(self.translation_history) > 10:
            self.translation_history = self.translation_history[:10]
        
        self.update_history_display()
    
    def update_history_display(self):
        """Update the history display"""
        if not self.translation_history:
            self.history_output.value = '<div style="text-align: center; color: #666; padding: 20px;">No translations yet. Start translating above!</div>'
            return
        
        history_html = '<div style="max-height: 300px; overflow-y: auto;">'
        
        for i, item in enumerate(self.translation_history):
            bg_color = "#f8f9fa" if i % 2 == 0 else "#ffffff"
            confidence_color = "#2ecc71" if item['confidence'] > 0.7 else "#f39c12" if item['confidence'] > 0.5 else "#e74c3c"
            
            history_html += f"""
            <div style="background: {bg_color}; padding: 12px; margin: 5px 0; border-radius: 8px; border-left: 4px solid {confidence_color};">
                <div style="display: flex; justify-content: space-between; align-items: center; font-size: 11px; color: #7f8c8d;">
                    <span>🕒 {item['timestamp']}</span>
                    <span style="background: {confidence_color}; color: white; padding: 2px 8px; border-radius: 10px;">
                        🎯 {item['confidence']:.1%}
                    </span>
                </div>
                <div style="font-size: 12px; margin: 8px 0; color: #2c3e50;">
                    <strong>EN:</strong> {item['english']}
                </div>
                <div style="font-size: 13px; color: #34495e; font-weight: 500;">
                    <strong>ML:</strong> {item['malayalam']}
                </div>
                <div style="font-size: 10px; color: #95a5a6; text-align: right; margin-top: 5px;">
                    📊 {item['characters']} characters
                </div>
            </div>
            """
        
        history_html += '</div>'
        self.history_output.value = history_html
    
    def update_session_stats(self):
        """Update session statistics"""
        if not self.translation_history:
            return
        
        total_translations = len(self.translation_history)
        total_chars = sum(item['characters'] for item in self.translation_history)
        avg_confidence = sum(item['confidence'] for item in self.translation_history) / total_translations
        session_duration = datetime.now() - self.session_start
        
        stats_html = f"""
        <div class="stats-card">
            <h4 style="margin: 0; color: #2c3e50;">📊 Session Statistics</h4>
            <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 10px; margin-top: 10px;">
                <div>
                    <div style="font-size: 24px; font-weight: bold; color: #3498db;">{total_translations}</div>
                    <div style="font-size: 12px;">Translations</div>
                </div>
                <div>
                    <div style="font-size: 24px; font-weight: bold; color: #e74c3c;">{total_chars}</div>
                    <div style="font-size: 12px;">Characters</div>
                </div>
                <div>
                    <div style="font-size: 24px; font-weight: bold; color: #2ecc71;">{avg_confidence:.1%}</div>
                    <div style="font-size: 12px;">Avg Confidence</div>
                </div>
                <div>
                    <div style="font-size: 24px; font-weight: bold; color: #9b59b6;">{session_duration.seconds//60}m</div>
                    <div style="font-size: 12px;">Session Time</div>
                </div>
            </div>
        </div>
        """
        
        self.stats_output.value = stats_html
    
    def show_notification(self, message, type="info"):
        """Show temporary notification"""
        colors = {
            "info": "#3498db",
            "success": "#2ecc71", 
            "warning": "#f39c12",
            "error": "#e74c3c"
        }
        
        notification_html = f"""
        <div class="notification" style="background: {colors[type]};">
            {message}
        </div>
        """
        
        # Display notification
        display(HTML(notification_html))
        
        # Use JavaScript to remove notification after delay
        display(Javascript("""
        setTimeout(function() {
            var notifications = document.querySelectorAll('.notification');
            notifications.forEach(function(notification) {
                notification.style.animation = 'slideOut 0.3s ease-in forwards';
                setTimeout(function() {
                    if (notification.parentNode) {
                        notification.parentNode.removeChild(notification);
                    }
                }, 300);
            });
        }, 3000);
        """))
    
    def show_error_display(self, error_message):
        """Show error display"""
        error_html = f"""
        <div class="translation-card" style="border-left-color: #e74c3c;">
            <div style="text-align: center; color: #e74c3c;">
                <h3>❌ Translation Error</h3>
                <p>{error_message}</p>
                <div style="background: #ffeaa7; padding: 15px; border-radius: 8px; margin-top: 15px; border-left: 4px solid #fdcb6e;">
                    <strong>💡 Pro Tip:</strong> Try shorter sentences or check your internet connection. 
                    The AI works best with clear, concise English text.
                </div>
            </div>
        </div>
        """
        self.malayalam_output.value = error_html
    
    def display_ui(self):
        """Display the complete UI"""
        clear_output(wait=True)
        display(self.main_container)
        
        # Show welcome notification
        self.show_notification("🚀 AI Translation Master is ready! Try translating some text!", "success")

# Initialize and display the advanced UI
print("🌐 Launching Advanced AI Translation Interface...")
print("🔄 Loading neural networks and preparing translation engine...")
advanced_ui = AdvancedTranslationUI(translator)

<IPython.core.display.Javascript object>

In [29]:
def final_comprehensive_demo():
    """Final demonstration showcasing all features"""
    print("=" * 80)
    print("🎯 FINAL ENGLISH-MALAYALAM TRANSLATION SYSTEM DEMONSTRATION")
    print("=" * 80)
    
    # Test categories
    categories = {
        "Greetings": [
            "Hello, how are you?",
            "Good morning, have a nice day!",
            "Thank you for your help"
        ],
        "Technology": [
            "Artificial intelligence is changing our world",
            "Python programming is very popular",
            "Data science requires statistics and programming skills"
        ],
        "Travel": [
            "Kerala is famous for its backwaters",
            "I want to visit Munnar hills",
            "Malayalam is the language of Kerala"
        ],
        "Business": [
            "Please send me the contract details",
            "We need to schedule a meeting next week",
            "The project deadline is approaching"
        ]
    }
    
    for category, sentences in categories.items():
        print(f"\n📂 {category.upper()} CATEGORY")
        print("-" * 60)
        
        for sentence in sentences:
            result = translator.translate_with_confidence(sentence)
            
            print(f"🔤 English: {sentence}")
            print(f"📜 Malayalam: {result['malayalam']}")
            print(f"📊 Confidence: {result['confidence']:.3f}")
            print()
    
    # Performance summary
    print("📈 PERFORMANCE SUMMARY")
    print("-" * 60)
    
    total_sentences = sum(len(sentences) for sentences in categories.values())
    confidence_scores = []
    
    for sentences in categories.values():
        for sentence in sentences:
            result = translator.translate_with_confidence(sentence)
            confidence_scores.append(result['confidence'])
    
    avg_confidence = sum(confidence_scores) / len(confidence_scores)
    
    print(f"Total sentences translated: {total_sentences}")
    print(f"Average confidence score: {avg_confidence:.3f}")
    print(f"Best confidence: {max(confidence_scores):.3f}")
    print(f"Worst confidence: {min(confidence_scores):.3f}")
    print(f"Model: Helsinki-NLP/opus-mt-en-ml")
    print(f"Device: {translator.device}")

# Run final demo
final_comprehensive_demo()

🎯 FINAL ENGLISH-MALAYALAM TRANSLATION SYSTEM DEMONSTRATION

📂 GREETINGS CATEGORY
------------------------------------------------------------
🔤 English: Hello, how are you?
📜 Malayalam: ഹലോ, സുഖമല്ലേ?
📊 Confidence: 0.698

🔤 English: Good morning, have a nice day!
📜 Malayalam: ഗുഡ് മോണിംഗ്, ശുഭദിനം!
📊 Confidence: 0.744

🔤 English: Thank you for your help
📜 Malayalam: നിങ്ങളുടെ സഹായത്തിന് നന്ദി.
📊 Confidence: 0.593


📂 TECHNOLOGY CATEGORY
------------------------------------------------------------
🔤 English: Artificial intelligence is changing our world
📜 Malayalam: അഗ്രകോടി ബുദ്ധി നമ്മുടെ ലോകം മാറുകയാണ്
📊 Confidence: 0.375

🔤 English: Python programming is very popular
📜 Malayalam: പൈത്തോൺ പ്രോഗ്രാമിങ്ങ് വളരെ ജനപ്രീതിയാർജിച്ചതാണ്
📊 Confidence: 0.604

🔤 English: Data science requires statistics and programming skills
📜 Malayalam: ഡാറ്റ ശാസ്ത്രത്തിന് സ്ഥിതിവിവരക്കണക്കുകളും പ്രോഗ്രാമിങ് വൈദഗ് ധ്യങ്ങളും ആവശ്യമാണ്
📊 Confidence: 0.545


📂 TRAVEL CATEGORY
-------------------------------------